# 2020–21 CRDC data cleaning

This notebook cleans the 2020–21 Civil Rights Data Collection (CRDC) school and LEA files using the sequence in `data_audit.md`.

2020–21 is the COVID year. Seclusion reporting is incomplete: 14,158 schools are DQ-suppressed (`-11`) and 4,700 virtual-only schools are skipped (`-13`). Use it for robustness / sensitivity analysis, not as the primary modeling year. Negative reserved codes are missingness, not counts — they are recoded to NA and never filled with 0. The COVID directional indicators are included as sensitivity covariates.

Cleaning sequence:
1. Read CSVs as strings (UTF-8, fall back to cp1252); drop empty rows.
2. Pad `LEAID` / `SCHID`, verify `COMBOKEY` = `LEAID` + `SCHID`, and inner-join the four school modules on `COMBOKEY`.
3. Left-join LEA characteristics on padded `LEAID`.
4. Recode reserved codes to NA; keep the original code and reason group; build seclusion instance totals, enrollment, and rates.
5. Flag implausible FTE values.
6. Flag JJ / special-education / virtual-only-skip / small-enrollment schools.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from clean_crdc_2020_21 import (
    COUNT_FTE_COLS,
    DIND_COLS,
    INSTRUCTION_TYPE_LABELS,
    FTE_ALL_COLS,
    LEA_COLS,
    LEA_PATH,
    OUT_DIR,
    RESERVED_GROUPS,
    SECL_INSTANCE_COLS,
    VIRTUAL_TYPE_LABELS,
    pad_id,
    print_audit_checks,
    read_crdc_csv,
    read_school_modules,
    recode_categorical,
    recode_reserved,
    unexpected_codes,
    yes_no_flag,
)

## 1. Read priority columns as strings and join school modules

2020–21 files are UTF-8 (the reader still falls back to `cp1252`). Every value is a string so IDs keep leading zeros. School Characteristics, Enrollment, Restraint and Seclusion, School Support, and COVID Directional Indicators share the same `COMBOKEY` set, so the inner join keeps every school.

In [ ]:
sch, join_info = read_school_modules()
lea = read_crdc_csv(LEA_PATH, LEA_COLS)
print(join_info)
sch.shape, lea.shape

## 2. Check `COMBOKEY`

Unlike 2015–16, the 2020–21 IDs are not corrupted. `COMBOKEY` is padded to 12 characters and should equal `LEAID` (7) + `SCHID` (5). Duplicates are checked before joining.

In [ ]:
lea["LEAID"] = pad_id(lea["LEAID"], 7)

print("COMBOKEY unique:", sch["COMBOKEY"].nunique(), "of", len(sch), "rows")
print("Duplicate keys:", sch["COMBOKEY"].duplicated().sum())
print("COMBOKEY != LEAID + SCHID:", (sch["LEAID"] + sch["SCHID"] != sch["COMBOKEY"]).sum())
print("Duplicate LEAID in LEA file:", lea["LEAID"].duplicated().sum())
unmatched = ~sch["LEAID"].isin(lea["LEAID"])
print("Schools with no LEA match:", int(unmatched.sum()))

## 3. Join school file to LEA characteristics

LEA fields (`LEA_ENR`, `LEA_SCHOOLS`, `CJJ`) are left-joined on `LEAID`. The Distance Education and High School Equivalency files are LEA-level extras outside the audit's priority fields, so they are not joined here.

In [ ]:
df = sch.merge(lea, on="LEAID", how="left", suffixes=("", "_lea"))
df.shape

## 4. Recode reserved codes; build outcomes

Values `< 0` are reserved codes (`-9` skip, `-13` virtual-only skip, `-11` DQ suppression, `-5` action plan, etc.). Recode them to NA and keep the original code and reason group. Zeros stay zeros.

Seclusion instances are summed only when WODIS, IDEA, and 504 are all valid. The rate is `100 * instances / (TOT_ENR_M + TOT_ENR_F)` and is NA when enrollment is missing or 0.

The COVID directional indicators (`SCH_DIND_INSTRUCTIONTYPE`, `SCH_DIND_VIRTUALTYPE`) are letter categories, so they keep their letters; reserved codes (`-3`, `-13`) become NA with the code and reason kept in `_code` / `_reason` columns. Labels follow the 2020–21 Appendix Workbook (instruction type: A in-person only, B virtual only, C hybrid, D no effect; virtual type: A students in school / teachers remote, B students remote, C both). `SCH_DIND_VIRTUALTYPE = -13` is a structural skip for schools with no virtual instruction (types A and D).

In [ ]:
df = recode_reserved(df, COUNT_FTE_COLS)
df = recode_categorical(df, DIND_COLS)  # letter categories; -3/-13 -> NA + code
extra = {}

secl_valid = df[SECL_INSTANCE_COLS].notna().all(axis=1)
extra["seclusion_instances"] = np.where(
    secl_valid, df[SECL_INSTANCE_COLS].sum(axis=1), np.nan
)
wodis_code = df["SCH_RSINSTANCES_SECL_WODIS_code"]
extra["seclusion_reason_code"] = wodis_code.fillna(
    df["SCH_RSINSTANCES_SECL_IDEA_code"]
).fillna(df["SCH_RSINSTANCES_SECL_504_code"])
extra["seclusion_reason"] = extra["seclusion_reason_code"].map(RESERVED_GROUPS)

enr_valid = df["TOT_ENR_M"].notna() & df["TOT_ENR_F"].notna()
extra["enrollment"] = np.where(enr_valid, df["TOT_ENR_M"] + df["TOT_ENR_F"], np.nan)
extra["seclusion_rate_per_100"] = np.where(
    (extra["enrollment"] > 0) & secl_valid,
    100.0 * extra["seclusion_instances"] / extra["enrollment"],
    np.nan,
)
extra["flag_missing_seclusion"] = ~secl_valid
extra["flag_missing_enrollment"] = ~enr_valid
extra["flag_zero_enrollment"] = extra["enrollment"] == 0
extra["flag_rate_undefined"] = (
    extra["flag_missing_seclusion"]
    | extra["flag_missing_enrollment"]
    | extra["flag_zero_enrollment"]
)

print("All three seclusion fields valid:", int(secl_valid.sum()))
print("Seclusion reason (any field):")
print(extra["seclusion_reason"].value_counts(dropna=False).to_string())
print("WODIS -9:", int((wodis_code == -9).sum()))
print("WODIS -11:", int((wodis_code == -11).sum()))
print("WODIS -13:", int((wodis_code == -13).sum()))
print("WODIS -5:", int((wodis_code == -5).sum()))
print("WODIS > 0:", int((df["SCH_RSINSTANCES_SECL_WODIS"] > 0).sum()))
print("IDEA > 0:", int((df["SCH_RSINSTANCES_SECL_IDEA"] > 0).sum()))
print("TOT_ENR_M usable:", int(df["TOT_ENR_M"].notna().sum()))
print("TOT_ENR_M = 0:", int((df["TOT_ENR_M"] == 0).sum()))
print("M + F = 0:", int((extra["enrollment"] == 0).sum()))
print("Negative codes outside the 2020-21 set:", unexpected_codes(df, COUNT_FTE_COLS) or "none")

## 5–6. FTE plausibility and school-type flags

Reserved FTE codes (for example `-9` security FTE at JJ schools) are already NA. FTE is flagged as an outlier when it exceeds enrollment, or when FTE per 100 students is above 20 (50 for teachers). The 999 placeholder pattern is specific to 2015–16 and is not applied here. Schools skipped with `-13` on the seclusion module (virtual-only) are flagged as `flag_virtual_only_skip`; `-13` on `SCH_DIND_VIRTUALTYPE` is flagged separately as `flag_dind_virtual_skip`. Every seclusion `-13` school has instruction type B (virtual only).

In [ ]:
enrollment = extra["enrollment"]

for prefix, m_col, f_col in [
    ("idea", "SCH_ENR_IDEA_M", "SCH_ENR_IDEA_F"),
    ("sec504", "SCH_ENR_504_M", "SCH_ENR_504_F"),
    ("lep", "SCH_ENR_LEP_M", "SCH_ENR_LEP_F"),
    ("secl_students_nonidea", "TOT_RS_NONIDEA_SECL_M", "TOT_RS_NONIDEA_SECL_F"),
    ("secl_students_idea", "TOT_RS_IDEA_SECL_M", "TOT_RS_IDEA_SECL_F"),
]:
    both = df[m_col].notna() & df[f_col].notna()
    extra[f"{prefix}_n"] = np.where(both, df[m_col] + df[f_col], np.nan)
    if prefix in {"idea", "sec504", "lep"}:
        extra[f"{prefix}_share"] = np.where(
            enrollment > 0, extra[f"{prefix}_n"] / enrollment, np.nan
        )

extra["flag_fte_outlier"] = False
teacher_cols = {"SCH_FTETEACH_TOT", "SCH_FTETEACH_CERT"}
for col in FTE_ALL_COLS:
    per_100 = np.where(enrollment > 0, 100.0 * df[col] / enrollment, np.nan)
    extra[f"{col}_per_100"] = per_100
    exceeds_enrollment = df[col].notna() & pd.notna(enrollment) & (df[col] > enrollment)
    cap = 50 if col in teacher_cols else 20
    outlier = exceeds_enrollment | (df[col].notna() & (per_100 > cap))
    extra[f"{col}_outlier"] = outlier
    extra["flag_fte_outlier"] = extra["flag_fte_outlier"] | outlier.fillna(False)

extra["flag_jj"] = yes_no_flag(df["JJ"])
extra["flag_sped"] = yes_no_flag(df["SCH_STATUS_SPED"])
extra["flag_charter"] = yes_no_flag(df["SCH_STATUS_CHARTER"])
extra["flag_alt"] = yes_no_flag(df["SCH_STATUS_ALT"])
extra["flag_magnet"] = yes_no_flag(df["SCH_STATUS_MAGNET"])
extra["flag_small_enrollment"] = pd.notna(enrollment) & (enrollment < 50)
extra["flag_virtual_only_skip"] = wodis_code.eq(-13)
extra["covid_instruction_type"] = df["SCH_DIND_INSTRUCTIONTYPE"].map(INSTRUCTION_TYPE_LABELS)
extra["covid_virtual_type"] = df["SCH_DIND_VIRTUALTYPE"].map(VIRTUAL_TYPE_LABELS)
extra["flag_covid_virtual_only"] = df["SCH_DIND_INSTRUCTIONTYPE"].eq("B")
extra["flag_covid_hybrid"] = df["SCH_DIND_INSTRUCTIONTYPE"].eq("C")
extra["flag_dind_virtual_skip"] = df["SCH_DIND_VIRTUALTYPE_code"].eq(-13)
extra["flag_dind_missing"] = (
    df["SCH_DIND_INSTRUCTIONTYPE"].isna() | df["SCH_DIND_VIRTUALTYPE"].isna()
)
extra["year"] = "2020-21"

df = pd.concat([df, pd.DataFrame(extra, index=df.index)], axis=1)
print("JJ schools:", int(df["flag_jj"].sum()))
print("SPED schools:", int(df["flag_sped"].sum()))
print("Enrollment < 50:", int(df["flag_small_enrollment"].sum()))
print("Virtual-only skip (-13):", int(df["flag_virtual_only_skip"].sum()))
print("DIND virtual -13 skip:", int(extra["flag_dind_virtual_skip"].sum()))
print("FTE outlier schools:", int(df["flag_fte_outlier"].sum()))

## Audit checks and export

Compare cleaned totals to the 2020–21 counts in the data audit, then write the analysis-ready table.

In [ ]:
jj = df["flag_jj"]
report = {
    "n_schools": len(df),
    "unique_combokey": df["COMBOKEY"].nunique(),
    "unmatched_lea": int(unmatched.sum()),
    "key_mismatch": join_info["key_mismatch"],
    "rows_lost_in_join": join_info["n_char"] - join_info["n_joined"],
    "jj_yes": int(jj.sum()),
    "seclusion_all_three_valid": int((~df["flag_missing_seclusion"]).sum()),
    "wodis_neg9": int((df["SCH_RSINSTANCES_SECL_WODIS_code"] == -9).sum()),
    "wodis_neg11": int((df["SCH_RSINSTANCES_SECL_WODIS_code"] == -11).sum()),
    "wodis_neg13": int((df["SCH_RSINSTANCES_SECL_WODIS_code"] == -13).sum()),
    "wodis_neg5": int((df["SCH_RSINSTANCES_SECL_WODIS_code"] == -5).sum()),
    "wodis_positive": int((df["SCH_RSINSTANCES_SECL_WODIS"] > 0).sum()),
    "idea_positive": int((df["SCH_RSINSTANCES_SECL_IDEA"] > 0).sum()),
    "enr_m_usable": int(df["TOT_ENR_M"].notna().sum()),
    "enr_f_usable": int(df["TOT_ENR_F"].notna().sum()),
    "enr_m_neg11": int((df["TOT_ENR_M_code"] == -11).sum()),
    "enr_f_neg11": int((df["TOT_ENR_F_code"] == -11).sum()),
    "enr_m_zero": int((df["TOT_ENR_M"] == 0).sum()),
    "enr_sum_zero": int((df["enrollment"] == 0).sum()),
    "jj_security_gua_neg9": int((df.loc[jj, "SCH_FTESECURITY_GUA_code"] == -9).sum()),
    "fte_outlier_rows": int(df["flag_fte_outlier"].sum()),
    "dind_instruction_neg3": int((df["SCH_DIND_INSTRUCTIONTYPE_code"] == -3).sum()),
    "dind_virtual_neg13": int((df["SCH_DIND_VIRTUALTYPE_code"] == -13).sum()),
    "dind_virtual_neg3": int((df["SCH_DIND_VIRTUALTYPE_code"] == -3).sum()),
    "dind_neg13_not_AD": int(
        (df["flag_dind_virtual_skip"] & ~df["SCH_DIND_INSTRUCTIONTYPE"].isin(["A", "D"])).sum()
    ),
    "secl_neg13_not_virtual_only": int(
        (df["flag_virtual_only_skip"] & ~df["flag_covid_virtual_only"]).sum()
    ),
    "unexpected_codes": unexpected_codes(df, COUNT_FTE_COLS + DIND_COLS),
}
print_audit_checks(report)

preview_cols = [
    "year", "COMBOKEY", "LEAID", "SCHID", "LEA_STATE", "SCH_NAME",
    "seclusion_instances", "enrollment", "seclusion_rate_per_100",
    "seclusion_reason", "flag_jj", "flag_sped", "flag_small_enrollment",
    "flag_virtual_only_skip", "covid_instruction_type", "covid_virtual_type",
    "flag_fte_outlier",
]
df[preview_cols].head()

In [ ]:
OUT_DIR.mkdir(exist_ok=True)
out_csv = OUT_DIR / "crdc_2020_21_analysis_ready.csv"
df.to_csv(out_csv, index=False)
print(f"Wrote {out_csv} ({len(df):,} rows, {df.shape[1]} columns)")